In [15]:
import os
import kagglehub
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, RobustScaler, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

### Download the dataset only if you don't have it

In [1]:
main_dir = "playground-series-s6e7"

In [4]:
try:
    train_data = pd.read_csv(main_dir + "/train.csv")
    test_data = pd.read_csv(main_dir + "/test.csv")
    sample_submission = pd.read_csv(main_dir + "/sample_submission.csv")
except FileNotFoundError:
    print("Failed to read data")

### Basic data display for viewing

In [5]:
train_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 690088 entries, 0 to 690087
Data columns (total 15 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   id                       690088 non-null  int64  
 1   health_condition         690088 non-null  str    
 2   sleep_duration           614089 non-null  float64
 3   heart_rate               682255 non-null  float64
 4   bmi                      676190 non-null  float64
 5   calorie_expenditure      637235 non-null  float64
 6   step_count               676172 non-null  float64
 7   exercise_duration        683187 non-null  float64
 8   water_intake             646611 non-null  float64
 9   diet_type                683187 non-null  str    
 10  stress_level             607277 non-null  str    
 11  sleep_quality            631757 non-null  str    
 12  physical_activity_level  653467 non-null  str    
 13  smoking_alcohol          661506 non-null  str    
 14  gender         

In [6]:
train_data.head()

,id,health_condition,sleep_duration,heart_rate,bmi,calorie_expenditure,step_count,exercise_duration,water_intake,diet_type,stress_level,sleep_quality,physical_activity_level,smoking_alcohol,gender
0,0,unhealthy,5.22,70.6,25.66,2174.0,1326.0,19.8,1.86,veg,high,average,sedentary,yes,female
1,1,at-risk,5.53,71.3,25.84,1966.0,9891.0,49.9,1.26,non-veg,low,average,moderate,yes,other
2,2,unhealthy,5.29,75.4,24.54,2688.0,14216.0,38.1,1.60,veg,high,poor,active,yes,male
3,3,unhealthy,4.70,77.2,23.13,2630.0,7174.0,59.9,2.02,veg,high,average,active,occasional,female
4,4,at-risk,7.23,73.4,28.44,2560.0,6584.0,46.0,2.25,veg,NaN,average,sedentary,NaN,male


In [7]:
train_data.describe()

,id,sleep_duration,heart_rate,bmi,calorie_expenditure,step_count,exercise_duration,water_intake
count,690088.00000,614089.000000,682255.000000,676190.000000,637235.000000,676172.000000,683187.000000,646611.000000
mean,345043.50000,6.992597,75.096504,22.984925,2226.084931,8615.953050,38.751456,2.188542
std,199211.39062,1.215407,8.175106,2.481787,347.532098,3929.399831,14.742189,0.518489
min,0.00000,3.000000,50.000000,16.000000,1200.000000,1002.000000,0.000000,0.500000
25%,172521.75000,6.160000,69.400000,21.320000,2053.000000,5389.000000,29.200000,1.840000
50%,345043.50000,6.990000,75.100000,22.990000,2241.000000,8856.000000,39.400000,2.170000
75%,517565.25000,7.810000,80.700000,24.660000,2456.000000,12114.000000,49.400000,2.500000
max,690087.00000,10.000000,107.700000,34.820000,3580.000000,14999.000000,99.800000,4.720000


In [8]:
train_data['health_condition'].value_counts(normalize=True)

health_condition
at-risk      0.858675
unhealthy    0.083647
fit          0.057678
Name: proportion, dtype: float64

### Drop the 'id' column which lacks a predictive property as it may influence model's prediction

In [9]:
train_data = train_data.drop(columns="id")

Drop the target column so that it doesn't leak the result into the prediction.

- X is the train_data without the health_condition column
- y is only the health_condition column of the train_data
- X_test is the test_data which already doesn't contain the health_condition column

In [14]:
X = train_data.drop(columns=["health_condition"])   # drop the target column
y = train_data["health_condition"]  # keep the target column

X_test = test_data.copy()   # test data doesn't contain health_condition

80/20 split for testing and verification data
- X_train is the feature training data
- X_val is validation data
- y_train is the resultant column
- y_val is the result validation data

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size = 0.2,
    random_state = 42,
    stratify = y
)

# print the shapes
print(f"X original shape: ", {X.shape})
print(f"X_train shape: ", {X_train.shape})
print(f"X_val shape: ", {X_val.shape})
print(f"y original shape: ", {y.shape})
print(f"y_train shape: ", {y_train.shape})
print(f"y_val shape: ", {y_val.shape})

X original shape:  {(690088, 13)}
X_train shape:  {(552070, 13)}
X_val shape:  {(138018, 13)}
y original shape:  {(690088,)}
y_train shape:  {(552070,)}
y_val shape:  {(138018,)}


As Sean's EDA showed, there are 3 classes of data:
- Numerical
- Categorical
- Ordinal

We create lists for each type of data as the preprocessor would have to process each class of data differently

In [36]:
# best approach is to use a dictionary for the ordinal categories wherein the
# key is the column name and the value is a list of the categories
ordinal_map = {
    'sleep_quality': ["poor", "average", "good"],
    "physical_activity_level": ["sedentary", "moderate", "active"],
    "stress_level": ["low", "medium", "high"]
}

ordinal_cols = list(ordinal_map.keys())
ordinal_categories = list(ordinal_map.values())

# split data between categorical, ordinal and numerical data
numerical_cols = X.select_dtypes(include=[np.number]).columns.tolist()
# This was found out from the EDA previously done by Sean
categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()

# remove the ordinal columns
categorical_cols = [col for col in categorical_cols if col not in ordinal_cols]
print(f"numerical_cols {numerical_cols}")
print(f"ordinal_cols {ordinal_cols}")
print(f"categorical cols {categorical_cols}")
print(f"Ordinal categories {ordinal_categories}")

numerical_cols ['sleep_duration', 'heart_rate', 'bmi', 'calorie_expenditure', 'step_count', 'exercise_duration', 'water_intake']
ordinal_cols ['sleep_quality', 'physical_activity_level', 'stress_level']
categorical cols ['diet_type', 'smoking_alcohol', 'gender']
Ordinal categories [['poor', 'average', 'good'], ['sedentary', 'moderate', 'active'], ['low', 'medium', 'high']]


C:\Users\pierc\AppData\Local\Temp\ipykernel_33588\3796387934.py:15: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()


We create the preprocessor pipeline that will handle missing value imputation, encoding, and scaling of data

In [39]:
preprocessor = ColumnTransformer(
    transformers=[
        # Numerical pipeline: median imputation → missing indicator → robust scaling
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median', add_indicator=True)),
            ('scaler', RobustScaler())
        ]), numerical_cols),

        # Ordinal pipeline: most-frequent imputation + OrdinalEncoder
        ('ord', Pipeline([
            ('imputer', SimpleImputer(strategy="most_frequent")),
            ('encoder', OrdinalEncoder(
                    categories=ordinal_categories,
                    handle_unknown="use_encoded_value",
                    unknown_value=-1
                )
            )
        ]), ordinal_cols),

        # Categorical pipeline: most-frequent imputation → one-hot encoding
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
        ]), categorical_cols)
    ]
)

print(preprocessor)

ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(add_indicator=True,
                                                                strategy='median')),
                                                 ('scaler', RobustScaler())]),
                                 ['sleep_duration', 'heart_rate', 'bmi',
                                  'calorie_expenditure', 'step_count',
                                  'exercise_duration', 'water_intake']),
                                ('ord',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('encoder',
                                                  OrdinalE...
                                                                              'moderate',
                            

Using the preprocessor on the data

In [ ]:
# use the preprocessor on the datasets
X_train_processed = preprocessor.fit_transform(X_train)
X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)


['sleep_duration_missing', 'heart_rate_missing', 'bmi_missing', 'calorie_expenditure_missing', 'step_count_missing', 'exercise_duration_missing', 'water_intake_missing']


In [52]:
ord_feature_names = preprocessor.named_transformers_['ord'].named_steps['encoder'].get_feature_names_out(ordinal_cols)
num_feature_names = numerical_cols
indicator_names = [f"{col}_missing" for col in numerical_cols]
cat_feature_names = preprocessor.named_transformers_['cat'].named_steps['encoder'].get_feature_names_out(categorical_cols)

print(ord_feature_names)
all_feature_names = list(num_feature_names) + list(indicator_names) + list(ord_feature_names) + list(cat_feature_names)

print(f"Feature count {len(all_feature_names)}")
print(f"Feature names {all_feature_names}")

['sleep_quality' 'physical_activity_level' 'stress_level']
Feature count 26
Feature names ['sleep_duration', 'heart_rate', 'bmi', 'calorie_expenditure', 'step_count', 'exercise_duration', 'water_intake', 'sleep_duration_missing', 'heart_rate_missing', 'bmi_missing', 'calorie_expenditure_missing', 'step_count_missing', 'exercise_duration_missing', 'water_intake_missing', 'sleep_quality', 'physical_activity_level', 'stress_level', 'diet_type_balanced', 'diet_type_non-veg', 'diet_type_veg', 'smoking_alcohol_no', 'smoking_alcohol_occasional', 'smoking_alcohol_yes', 'gender_female', 'gender_male', 'gender_other']


In [53]:
# turn the processed data into dataframes
X_train_processed_df = pd.DataFrame(X_train_processed, columns=all_feature_names)
X_val_processed_df = pd.DataFrame(X_val_processed, columns=all_feature_names)
X_test_processed_df = pd.DataFrame(X_train_processed, columns=all_feature_names)

print(f"X_train_processed dataframe {X_train_processed_df.shape}")
print(f"X_val_processed dataframe {X_val_processed_df.shape}")
print(f"X_test_processed dataframe {X_test_processed_df.shape}")

X_train_processed dataframe (552070, 26)
X_val_processed dataframe (138018, 26)
X_test_processed dataframe (552070, 26)


These next cells are just to view the preprocessed data

In [ ]:
# checking if preprocessed data contains any NaNs
nan_count_train = np.isnan(X_train_processed).sum()
nan_count_val = np.isnan(X_val_processed).sum()
nan_count_test = np.isnan(X_test_processed).sum()

print(f"nan count in X_train_processed is {nan_count_train}")
print(f"nan count in X_val_processed is {nan_count_val}")
print(f"nan count in X_test is {nan_count_test}")

nan count in X_train_processed is 0
nan count in X_val_processed is 0
nan count in X_test is 0


In [ ]:
# print every column in the X_train_processed dataframe
pd.set_option("display.max_columns", None)
X_train_processed_df.head()

,sleep_duration,heart_rate,bmi,calorie_expenditure,step_count,exercise_duration,water_intake,sleep_duration_missing,heart_rate_missing,bmi_missing,calorie_expenditure_missing,step_count_missing,exercise_duration_missing,water_intake_missing,sleep_quality,physical_activity_level,stress_level,diet_type_balanced,diet_type_non-veg,diet_type_veg,smoking_alcohol_no,smoking_alcohol_occasional,smoking_alcohol_yes,gender_female,gender_male,gender_other
0,0.130435,1.459459,1.190769,1.067568,-1.124887,0.485,-0.233333,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
1,0.000000,0.000000,0.381538,0.383784,0.695324,0.570,-0.566667,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,2.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
2,1.210145,0.657658,-0.486154,0.200000,0.093062,-0.805,1.533333,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0
3,0.000000,-0.054054,-0.123077,-0.202703,-0.272097,-0.890,-0.416667,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0
4,1.333333,-0.621622,-0.301538,-0.356757,0.703620,0.675,0.300000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,2.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0


In [61]:
# check medians and IQRs after scaling
X_train_processed_df[numerical_cols].describe().loc[["mean", "std", "50%"]]

,sleep_duration,heart_rate,bmi,calorie_expenditure,step_count,exercise_duration,water_intake
mean,0.002019,0.000277,-0.001941,-0.034629,-0.035756,-0.031729,0.029001
std,0.830769,0.732752,0.755792,0.903033,0.586871,0.733368,0.836906
50%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


In [63]:
print(y_train.value_counts(normalize=True))
print(train_data['health_condition'].value_counts(normalize=True))

health_condition
at-risk      0.858676
unhealthy    0.083647
fit          0.057677
Name: proportion, dtype: float64
health_condition
at-risk      0.858675
unhealthy    0.083647
fit          0.057678
Name: proportion, dtype: float64


Encode the y_train data which contains only the health_condition from the train_data

In [ ]:
label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)
y_val_encoded = label_encoder.transform(y_val)

print(label_encoder.classes_)



['at-risk' 'fit' 'unhealthy']
[0 0 0 0 0]
